In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
import glob
import pandas as pd
from datetime import datetime
from sunpy.coordinates import sun

from view import *

In [2]:
AU = 149597870691.

df = pd.read_csv('wcs.csv', skipinitialspace=True).dropna()

dids = df['did'].to_numpy()
dates = np.array([datetime.fromisoformat(date) for date in df['date']])
sec = np.array([(datetime.fromisoformat(date) - dates[0]).total_seconds() for date in df['date']])
days = sec / 24 / 3600

crln = df['crln_obs'].to_numpy()
crlt = df['crlt_obs'].to_numpy()

obs_vw = df['obs_vw'].to_numpy()
obs_vn = df['obs_vn'].to_numpy()
obs_vr = df['obs_vr'].to_numpy()

dsun = df['dsun_au'].to_numpy() * AU

In [3]:
rot = ~Rotate.x(crlt) + Rotate.y(crln)

r = rot((0, 0, 1))[0]
q = rot((-obs_vn, obs_vw, 0))[0]

r = np.array(r).T
q = np.array(q).T

In [4]:
def ecc_anomaly(theta, e):
    cos_f = np.cos(theta * np.pi / 180)
    sin_f = np.sin(theta * np.pi / 180)
    return np.arctan2(np.sqrt(1 - e ** 2) * sin_f, e + cos_f) * 180 / np.pi % 360 + theta // 360 * 360


def orbital_time(theta, a, e, alpha=1.327124e20):
    E = ecc_anomaly(theta, e) * np.pi / 180
    return a * np.sqrt(a / alpha) * (E - e * np.sin(E))


def kepler(r, vt, vr, alpha=1.327124e20):
    p = (r * vt) ** 2 / alpha
    v0 = np.sqrt(alpha / p)
    e = np.sqrt((vt / v0 - 1) ** 2 + (vr / v0) ** 2)
    a = p / (1 - e ** 2)

    theta = np.arctan2(vr, vt - v0) * 180 / np.pi % 360
    return theta, a, e

In [9]:
i = -180

theta, a, e = kepler(dsun[i], np.sqrt(obs_vw[i] ** 2 + obs_vn[i] ** 2), obs_vr[i])
t = orbital_time(theta + 360, a, e)

phi = np.arange(-180, 180, 0.1)
theta1 = theta + phi

t1 = orbital_time(theta1 + 360, a, e)
r1 = Rotate.from_axis_angle(q[i], phi)(r[i])[0]

dt = (t1 - t) / 24 / 3600

In [10]:
WSID = 360 / 25.38
crln_ = (np.arctan2(r1[0], r1[2]) * 180 / np.pi - dt * WSID) % 360

plt.figure(figsize=(10,8))
plt.plot(dt, crln_, '.', ms=10)
plt.plot(days - days[i], crln, '.')

plt.tight_layout()

In [11]:
WSID = 360 / 25.38

crln_ = (np.arctan2(r1[0], r1[2]) * 180 / np.pi - dt * WSID) % 360
for k in np.where(crln_[1:] > crln_[:-1])[0]:
    crln_[k+1:] -= 360
crln_ -= crln_[len(crln_) // 2]

x = np.arange(-180, 180)
y = np.interp(-x, -crln_, dt)

plt.figure(figsize=(10,8))
plt.plot(x, y)
plt.tight_layout()